In [ ]:
import os, re, glob, random, warnings, hashlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from lightgbm import LGBMClassifier, early_stopping

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

warnings.filterwarnings("ignore")

for old_f in ["submission.csv", "/kaggle/working/submission.csv"]:
    if os.path.exists(old_f):
        os.remove(old_f)

# =====================================================================
# 0. CONFIGURATION (LOCKED TO V32 0.67681 HUM-NOTCH CORE)
# =====================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running V33-HumNotchPro (Built on V32 0.67681 Core + 60-Model 2x5-Fold Bagging) on: {DEVICE}")

EMG_COLS = [f'ch{i}' for i in range(1, 9)]
IMU_COLS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
SIGNAL_COLS = EMG_COLS + IMU_COLS
AUX_REG_COLS = [
    'roll', 'pitch', 'yaw', 'thumb_curl', 'index_curl', 'middle_curl',
    'ring_curl', 'pinky_curl', 'pinch', 'openness', 'thumb_up'
]
NUM_REG = len(AUX_REG_COLS)

GESTURES = ['rest', 'fist', 'open', 'point', 'pinch', 'thumb-up']
GESTURE_MAP = {g: i for i, g in enumerate(GESTURES)}
INV_GESTURE_MAP = {i: g for i, g in enumerate(GESTURES)}

SEQ_LEN = 192
WINDOW_SEC = 1.4
TRAIN_STRIDE_SEC = 0.40
BATCH_SIZE = 64
EPOCHS = 36
SNAP_WEIGHTS = {34: 0.25, 35: 0.35, 36: 0.40}
LR = 1.1e-3
N_FOLDS = 5

SPLIT_CONFIGS = [
    {'shuffle': False, 'random_state': None, 'seed_base': 42},
    {'shuffle': True,  'random_state': 2026, 'seed_base': 2026},
]

DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.startswith('train_') and f.endswith('.xlsx') for f in files):
        DATA_DIR = os.path.dirname(root) if os.path.basename(root) == 'train' else root
        break
if DATA_DIR is None:
    DATA_DIR = '/kaggle/input/competitions/vr-calibration-free-emg-challenge-pieeg'

def natural_key(path):
    nums = re.findall(r'\d+', os.path.basename(path))
    return int(nums[0]) if nums else 0

def read_aura_excel_fast(fpath):
    xl = pd.ExcelFile(fpath, engine='openpyxl')
    sheet = 'multimodal' if 'multimodal' in xl.sheet_names else xl.sheet_names[0]
    df = xl.parse(sheet)
    df.columns = [str(c).replace('#', '').strip() for c in df.columns]
    return fpath, df

# =====================================================================
# 1. REFLECT-PADDED 50/60/100/120HZ NOTCH + IMU INTERPOLATION + SLOW BULGE
# =====================================================================
def get_window_duration(df_win):
    if 't' in df_win.columns and len(df_win) >= 2:
        dur = float(df_win['t'].iloc[-1] - df_win['t'].iloc[0])
        if 0.6 <= dur <= 2.2:
            return dur
    return 1.38

def reflect_gaussian_mains_notch(emg_clean, dur):
    """
    Reflect-padded C-infinity Gaussian notch at 50/60/100/120 Hz using exact per-window
    sampling rate fs = T / dur -> zero circular FFT wrap-around, zero Gibbs ringing.
    """
    T = emg_clean.shape[0]
    if T < 32:
        return emg_clean
    pad = min(48, T - 1)
    padded = np.pad(emg_clean, ((pad, pad), (0, 0)), mode='reflect')
    T_pad = padded.shape[0]

    freqs = np.fft.rfftfreq(T_pad, d=dur / float(T)).astype(np.float32)
    gain_f = np.ones_like(freqs, dtype=np.float32)
    for f0, bw in [(50.0, 2.2), (60.0, 2.2), (100.0, 2.2), (120.0, 2.2)]:
        gain_f *= (1.0 - np.exp(-0.5 * ((freqs - f0) / bw) ** 2))

    spec = np.fft.rfft(padded.astype(np.float32), axis=0)
    notched_pad = np.fft.irfft(spec * gain_f[:, None], n=T_pad, axis=0).astype(np.float32)
    emg_notched = notched_pad[pad : pad + T].copy()
    emg_notched -= np.mean(emg_notched, axis=0, keepdims=True)
    return emg_notched

def detrend_window_clean(df_win):
    """
    Upgrades V32 (0.67681) detrending cleanly:
      1. Linearly interpolates frozen imu_new == 0 step-hold rows before smoothing.
      2. Preserves V32's k_w=21 moving-average separation AND saves the calibrated low-freq muscle bulge (emg_slow).
      3. Applies reflect-padded 50/60/100/120 Hz Gaussian notch to emg_clean.
    """
    raw = df_win[SIGNAL_COLS].astype(np.float32).values.copy()
    T = raw.shape[0]
    dur = get_window_duration(df_win)

    # Fix 2/3 frozen IMU staircase step-holds (train imu_new=0.365 vs test imu_new=0.336)
    if 'imu_new' in df_win.columns and T > 6:
        imu_mask = (pd.to_numeric(df_win['imu_new'], errors='coerce').fillna(1).values > 0.5)
        if np.sum(imu_mask) >= 4:
            raw[~imu_mask, 8:] = np.nan

    raw = pd.DataFrame(raw).interpolate(method='linear', limit_direction='both').fillna(0.0).values

    centered = raw - np.mean(raw, axis=0, keepdims=True)
    if T > 4:
        t_axis = np.linspace(-1.0, 1.0, T, dtype=np.float32)[:, None]
        slope = np.sum(centered * t_axis, axis=0, keepdims=True) / (np.sum(t_axis ** 2) + 1e-8)
        detrended = centered - slope * t_axis
    else:
        detrended = centered

    emg_raw = detrended[:, :8]
    k_w = min(21, max(5, T // 10))
    emg_slow = pd.DataFrame(emg_raw).rolling(window=k_w, min_periods=1, center=True).mean().values.astype(np.float32)
    emg_clean = (emg_raw - emg_slow).astype(np.float32)
    emg_clean = reflect_gaussian_mains_notch(emg_clean, dur)

    imu_clean = pd.DataFrame(detrended[:, 8:]).rolling(window=5, min_periods=1, center=True).mean().values
    imu_clean = (imu_clean - np.mean(imu_clean, axis=0, keepdims=True)).astype(np.float32)
    return np.concatenate([emg_clean, imu_clean], axis=1).astype(np.float32), emg_slow, dur

def compute_session_channel_gains(window_dfs):
    rms_list, slow_list = [], []
    for w in window_dfs:
        sig, emg_slow, _ = detrend_window_clean(w)
        rms_list.append(np.sqrt(np.mean(sig ** 2, axis=0)) + 1e-5)
        slow_list.append(np.sqrt(np.mean(emg_slow ** 2, axis=0)) + 1e-5)
    ch_gain = (np.percentile(np.stack(rms_list), 60, axis=0) + 1e-4).astype(np.float32)
    slow_gain = (np.percentile(np.stack(slow_list), 60, axis=0) + 1e-4).astype(np.float32)
    return ch_gain, slow_gain

def extract_v33_features(df_win, session_ch_gain, slow_ch_gain, seq_len=SEQ_LEN):
    sig_uncal, emg_slow_u, dur = detrend_window_clean(df_win)
    T = sig_uncal.shape[0]

    sig_cal = sig_uncal / session_ch_gain[None, :]
    emg_cal, imu_cal = sig_cal[:, :8], sig_cal[:, 8:]
    emg_slow_cal = emg_slow_u / slow_ch_gain[None, :]

    # Stream A: Exact V1 36-Ch Z-Scored Waveform (hum-free & IMU-interpolated)
    delta_uncal = np.diff(sig_uncal, axis=0, prepend=sig_uncal[:1, :])
    emg_u = sig_uncal[:, :8]
    emg_p = np.roll(emg_u, 1, axis=0); emg_p[0] = emg_u[0]
    emg_n = np.roll(emg_u, -1, axis=0); emg_n[-1] = emg_u[-1]
    tkeo_u = (emg_u ** 2) - (emg_p * emg_n)

    v1_raw = np.concatenate([sig_uncal, delta_uncal, tkeo_u], axis=1)
    old_idx, new_idx = np.linspace(0.0, 1.0, T), np.linspace(0.0, 1.0, seq_len)
    v1_res = np.column_stack([np.interp(new_idx, old_idx, v1_raw[:, c]) for c in range(36)]).astype(np.float32)
    v1_norm = np.clip((v1_res - np.mean(v1_res, axis=0, keepdims=True)) / (np.std(v1_res, axis=0, keepdims=True) + 1e-5), -8.0, 8.0)

    # Stream B: Exact V3 30-Ch Impedance-Calibrated Spatial & Envelope Stream
    env_k = max(5, T // 14)
    emg_env_cal = np.sqrt(pd.DataFrame(emg_cal ** 2).rolling(window=env_k, min_periods=1, center=True).mean().values + 1e-6)
    spatial_prop_cal = emg_env_cal / (np.sum(emg_env_cal, axis=1, keepdims=True) + 1e-4)
    v3_raw = np.concatenate([
        np.sign(emg_cal) * np.log1p(np.abs(emg_cal)),
        np.log1p(emg_env_cal),
        spatial_prop_cal,
        np.sign(imu_cal) * np.log1p(np.abs(imu_cal))
    ], axis=1)
    v3_res = np.column_stack([np.interp(new_idx, old_idx, v3_raw[:, c]) for c in range(30)]).astype(np.float32)
    seq_tensor = np.concatenate([v1_norm, np.clip(v3_res, -8.0, 8.0)], axis=1).T.astype(np.float32)

    # Exact V32 (0.67681) Tabular Core + Clean TKEO Recruitment & Mechanical Muscle-Bulge Profiles
    feats = []
    emg_diff_cal = np.diff(emg_cal, axis=0, prepend=emg_cal[:1, :])
    rms_cal = np.sqrt(np.mean(emg_cal ** 2, axis=0)) + 1e-5
    mav_cal = np.mean(np.abs(emg_cal), axis=0) + 1e-5
    wl_cal = np.mean(np.abs(emg_diff_cal), axis=0) + 1e-5
    p90_cal = np.percentile(np.abs(emg_cal), 90, axis=0) + 1e-5
    p50_cal = np.percentile(np.abs(emg_cal), 50, axis=0) + 1e-5
    zc_cal = np.sum(np.diff(np.sign(emg_cal), axis=0) != 0, axis=0) / float(T)

    # Cols 0..47: Core EMG Amplitudes & ZC
    feats.extend(np.log1p(rms_cal)); feats.extend(np.log1p(mav_cal)); feats.extend(np.log1p(wl_cal))
    feats.extend(np.log1p(p90_cal)); feats.extend(np.log1p(p50_cal)); feats.extend(zc_cal)

    # Cols 48..87: Core Spatial Proportions
    rms_prop = rms_cal / np.sum(rms_cal)
    feats.extend(rms_prop); feats.extend(wl_cal / np.sum(wl_cal))
    feats.extend(p90_cal / np.sum(p90_cal)); feats.extend(mav_cal / np.sum(mav_cal))
    rms_u = np.sqrt(np.mean(emg_u ** 2, axis=0)) + 1e-5
    feats.extend(rms_u / np.sum(rms_u))

    # NEW (+16): Calibrated Hum-Free TKEO Recruitment Power (Amplitude^2 * Freq^2)
    ec_p = np.roll(emg_cal, 1, axis=0); ec_p[0] = emg_cal[0]
    ec_n = np.roll(emg_cal, -1, axis=0); ec_n[-1] = emg_cal[-1]
    tkeo_cal_e = np.mean(np.abs((emg_cal ** 2) - (ec_p * ec_n)), axis=0) + 1e-5
    feats.extend(np.log1p(tkeo_cal_e))
    feats.extend(tkeo_cal_e / np.sum(tkeo_cal_e))

    # NEW (+16): Calibrated Low-Frequency Mechanical Muscle-Bulge Profile (isolated from emg_cal)
    rms_slow_cal = np.sqrt(np.mean(emg_slow_cal ** 2, axis=0)) + 1e-5
    feats.extend(np.log1p(rms_slow_cal))
    feats.extend(rms_slow_cal / np.sum(rms_slow_cal))

    tot_rms_cal = float(np.log1p(np.sum(rms_cal)))
    tot_wl_cal = float(np.log1p(np.sum(wl_cal)))
    entropy = float(-np.sum(rms_prop * np.log(rms_prop + 1e-6)))
    feats.extend([
        tot_rms_cal, tot_wl_cal, entropy, float(np.max(rms_prop)), float(np.min(rms_prop)),
        float(np.std(rms_prop)), float(np.log1p(np.max(rms_cal))), float(np.log1p(np.median(rms_cal))),
        float(np.max(rms_prop) / (np.median(rms_prop) + 1e-4)), float(np.argmax(rms_prop))
    ])

    for i in range(8):
        feats.append(float(rms_prop[i] - rms_prop[(i + 4) % 8]))
        feats.append(float(rms_prop[i] - rms_prop[(i + 1) % 8]))
        feats.append(float(np.log1p(rms_cal[i]) - np.log1p(rms_cal[(i + 4) % 8])))

    cov_cal = (emg_cal.T @ emg_cal) / float(T) + 1e-3 * np.eye(8)
    eigvals_c, eigvecs_c = np.linalg.eigh(cov_cal)
    eigvals_c = np.clip(eigvals_c, 1e-4, None)
    log_cov_cal = eigvecs_c @ np.diag(np.log(eigvals_c)) @ eigvecs_c.T
    feats.extend(log_cov_cal[np.triu_indices(8, k=0)])
    feats.extend(np.log(eigvals_c))
    corr_mat = np.nan_to_num(np.corrcoef(emg_cal.T), nan=0.0)
    feats.extend(corr_mat[np.triu_indices(8, k=1)])

    for sub in np.array_split(emg_env_cal, 3, axis=0):
        s_m = np.mean(sub, axis=0) + 1e-5
        feats.extend(np.log1p(s_m)); feats.extend(s_m / np.sum(s_m))
    feats.extend(np.percentile(emg_env_cal, 90, axis=0) - np.percentile(emg_env_cal, 20, axis=0))
    feats.extend(np.mean(emg_env_cal[T // 4 : 3 * T // 4], axis=0) / (np.mean(emg_env_cal[:T // 4], axis=0) + 1e-3))

    # Physical Hz Spectral Bands (Skips 47-63 Hz mains zone completely)
    freqs = np.fft.rfftfreq(T, d=dur / float(T))
    fft_p = np.abs(np.fft.rfft(emg_cal, axis=0)) ** 2
    m1 = (freqs >= 12.0) & (freqs < 30.0)
    m2 = (freqs >= 30.0) & (freqs < 46.0)
    m3 = (freqs >= 64.0) & (freqs <= 92.0)
    if np.any(m1) and np.any(m2) and np.any(m3):
        p1 = np.mean(fft_p[m1], axis=0) + 1e-5
        p2 = np.mean(fft_p[m2], axis=0) + 1e-5
        p3 = np.mean(fft_p[m3], axis=0) + 1e-5
        ptot = p1 + p2 + p3
        feats.extend(p1 / ptot); feats.extend(p2 / ptot); feats.extend(p3 / ptot)
    else:
        feats.extend(np.zeros(24))

    imu_rms_cal = np.sqrt(np.mean(imu_cal ** 2, axis=0)) + 1e-5
    feats.extend(np.log1p(imu_rms_cal))
    feats.extend(np.log1p(np.percentile(np.abs(imu_cal), 90, axis=0)))
    feats.extend(np.percentile(imu_cal, 10, axis=0))
    feats.extend(np.percentile(imu_cal, 90, axis=0))
    feats.extend(imu_rms_cal[:3] / (np.sum(imu_rms_cal[:3]) + 1e-5))
    feats.extend(imu_rms_cal[3:] / (np.sum(imu_rms_cal[3:]) + 1e-5))

    imu_traj = np.cumsum(imu_cal, axis=0) / float(T)
    feats.extend(np.log1p(np.std(imu_traj, axis=0)))
    feats.extend(np.mean(imu_cal[T // 2:], axis=0) - np.mean(imu_cal[:T // 2], axis=0))

    acc_mag = np.sqrt(np.sum(imu_cal[:, :3] ** 2, axis=1))
    gyro_mag = np.sqrt(np.sum(imu_cal[:, 3:] ** 2, axis=1))
    acc_jerk = np.sqrt(np.sum(np.diff(imu_cal[:, :3], axis=0, prepend=imu_cal[:1, :3]) ** 2, axis=1))

    feats.extend([
        float(np.log1p(np.mean(acc_mag))), float(np.log1p(np.std(acc_mag))),
        float(np.log1p(np.mean(gyro_mag))), float(np.log1p(np.std(gyro_mag))),
        float(np.log1p(np.mean(acc_jerk))), float(np.std(acc_jerk)),
        float(np.log1p(np.sum(rms_cal)) / (np.log1p(np.mean(gyro_mag)) + 0.1))
    ])
    imu_corr = np.nan_to_num(np.corrcoef(imu_cal.T), nan=0.0)
    feats.extend(imu_corr[np.triu_indices(6, k=1)])

    tab_vec = np.nan_to_num(np.array(feats, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return seq_tensor, tab_vec

def append_session_aligned_features(X_tab, session_ids):
    N, D = X_tab.shape
    X_sess_z = np.zeros_like(X_tab)
    X_sess_rank = np.zeros((N, 54), dtype=np.float32)

    for s_id in np.unique(session_ids):
        mask = (session_ids == s_id)
        if np.sum(mask) < 3:
            continue
        sub = X_tab[mask]
        med = np.median(sub, axis=0, keepdims=True)
        iqr = (np.percentile(sub, 75, axis=0, keepdims=True) - np.percentile(sub, 25, axis=0, keepdims=True)) + 1e-4
        X_sess_z[mask] = np.clip((sub - med) / iqr, -6.0, 6.0)

        for c in range(54):
            vals = sub[:, c]
            X_sess_rank[mask, c] = (np.argsort(np.argsort(vals)).astype(np.float32) / float(max(1, len(vals) - 1))) - 0.5

    return np.concatenate([X_tab, X_sess_z, X_sess_rank], axis=1)

# =====================================================================
# 2. PARALLEL DATA LOADER WITH OPTICAL-AGREEMENT LABEL NOISE WEIGHTING
# =====================================================================
def process_single_train_session(args):
    session_idx, (fpath, df) = args
    if 'label_source' in df.columns:
        df = df[df['label_source'].astype(str).str.strip().str.lower() == 'guided'].reset_index(drop=True)
    label_col = 'label' if 'label' in df.columns else 'gesture'
    df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    df = df[df[label_col].isin(GESTURE_MAP.keys())].reset_index(drop=True)
    if len(df) == 0:
        return None

    df['seg_id'] = ((df['t'].diff().abs() > 0.25) | (df[label_col] != df[label_col].shift())).cumsum()
    raw_windows, cls_list, reg_list, mask_list, anchor_list, qual_list = [], [], [], [], [], []

    for _, seg in df.groupby('seg_id'):
        t_start, t_end = seg['t'].iloc[0], seg['t'].iloc[-1]
        seg_label_str = seg[label_col].iloc[0]
        gesture_id = GESTURE_MAP[seg_label_str]
        curr_t, step_idx = t_start, 0
        while curr_t + WINDOW_SEC <= t_end + 0.08:
            win = seg[(seg['t'] >= curr_t) & (seg['t'] < curr_t + WINDOW_SEC)]
            if len(win) >= 35:
                r_t, r_m = np.zeros(NUM_REG, dtype=np.float32), np.zeros(NUM_REG, dtype=np.float32)
                for idx, col in enumerate(AUX_REG_COLS):
                    if col in win.columns:
                        vals = pd.to_numeric(win[col], errors='coerce').dropna().values
                        if len(vals) > 0:
                            r_t[idx], r_m[idx] = float(np.mean(vals)), 1.0

                # Softly down-weight reaction-lag windows where high-confidence camera clearly sees another gesture
                w_qual = 1.0
                if 'optical_present' in win.columns and 'optical_class' in win.columns and 'optical_conf' in win.columns:
                    op_pres = pd.to_numeric(win['optical_present'], errors='coerce').fillna(0).values
                    op_conf = pd.to_numeric(win['optical_conf'], errors='coerce').fillna(0).values
                    valid_op = (op_pres > 0.5) & (op_conf >= 0.75)
                    if np.sum(valid_op) >= max(5, len(win) // 5):
                        op_vals = [c for c in win['optical_class'].astype(str).str.strip().str.lower().values[valid_op] if c in GESTURE_MAP]
                        if len(op_vals) >= 5 and float(np.mean([c == seg_label_str for c in op_vals])) < 0.25:
                            w_qual = 0.35

                raw_windows.append(win); cls_list.append(gesture_id)
                reg_list.append(r_t); mask_list.append(r_m)
                anchor_list.append(1 if (step_idx % 3 == 0) else 0)
                qual_list.append(w_qual)
                step_idx += 1
            curr_t += TRAIN_STRIDE_SEC

    if not raw_windows:
        return None
    anchor_wins = [w for w, is_a in zip(raw_windows, anchor_list) if is_a == 1]
    ch_gain, slow_gain = compute_session_channel_gains(anchor_wins if len(anchor_wins) >= 5 else raw_windows)
    pairs = [extract_v33_features(win, ch_gain, slow_gain, seq_len=SEQ_LEN) for win in raw_windows]
    seq_list, tab_list = zip(*pairs)
    n = len(seq_list)
    return (
        np.stack(seq_list), np.stack(tab_list), np.array(cls_list, dtype=np.int64),
        np.stack(reg_list), np.stack(mask_list),
        np.full(n, session_idx, dtype=np.int64), np.array(anchor_list, dtype=np.int64),
        np.array(qual_list, dtype=np.float32)
    )

def load_v33_datasets(data_dir):
    train_files = sorted(glob.glob(os.path.join(data_dir, "**", "train_*.xlsx"), recursive=True), key=natural_key)
    test_files = sorted(glob.glob(os.path.join(data_dir, "**", "test_*.xlsx"), recursive=True), key=natural_key)
    print(f"Parallel-loading {len(train_files)} train files and {len(test_files)} test files...")

    with ThreadPoolExecutor(max_workers=4) as ex:
        train_loaded = list(ex.map(read_aura_excel_fast, train_files))
        test_loaded = list(ex.map(read_aura_excel_fast, test_files))
        train_res = [r for r in ex.map(process_single_train_session, enumerate(train_loaded)) if r is not None]

    X_seq = np.concatenate([r[0] for r in train_res], axis=0)
    X_tab = np.concatenate([r[1] for r in train_res], axis=0)
    y_cls = np.concatenate([r[2] for r in train_res], axis=0)
    y_reg = np.concatenate([r[3] for r in train_res], axis=0)
    reg_mask = np.concatenate([r[4] for r in train_res], axis=0)
    groups = np.concatenate([r[5] for r in train_res], axis=0)
    is_anchor = np.concatenate([r[6] for r in train_res], axis=0)
    sample_qual = np.concatenate([r[7] for r in train_res], axis=0)

    for j in range(y_reg.shape[1]):
        valid = reg_mask[:, j] > 0.5
        if np.sum(valid) > 10:
            y_reg[valid, j] = np.clip((y_reg[valid, j] - np.mean(y_reg[valid, j])) / (np.std(y_reg[valid, j]) + 1e-5), -4.0, 4.0)

    X_test_seq, X_test_tab, test_ids, test_sessions = [], [], [], []
    for test_sess_idx, (fpath, df) in enumerate(test_loaded):
        id_col = 'action_id' if 'action_id' in df.columns else df.columns[0]
        sess_wins, sess_ids = [], []
        for act_id, win in df.groupby(id_col, sort=False):
            sess_wins.append(win); sess_ids.append(act_id)
        test_ch_gain, test_slow_gain = compute_session_channel_gains(sess_wins)
        for act_id, win in zip(sess_ids, sess_wins):
            s_t, t_v = extract_v33_features(win, test_ch_gain, test_slow_gain, seq_len=SEQ_LEN)
            X_test_seq.append(s_t); X_test_tab.append(t_v)
            test_ids.append(act_id); test_sessions.append(test_sess_idx)

    X_test_seq, X_test_tab = np.stack(X_test_seq), np.stack(X_test_tab)
    test_sessions = np.array(test_sessions, dtype=np.int64)

    X_tab = append_session_aligned_features(X_tab, groups)
    X_test_tab = append_session_aligned_features(X_test_tab, test_sessions)

    valid_cols = np.std(X_tab, axis=0) > 1e-5
    X_tab = X_tab[:, valid_cols]
    X_test_tab = X_test_tab[:, valid_cols]

    n_down = int(np.sum(sample_qual < 0.99))
    print(f"Train Windows: {len(X_seq)} (Anchors: {np.sum(is_anchor)} | Optical-Mismatch Downweighted: {n_down} [{100.0*n_down/len(X_seq):.1f}%])")
    print(f"Seq: {X_seq.shape} | Clean Tab: {X_tab.shape} | Test Actions: {len(X_test_seq)}")
    return X_seq, X_tab, y_cls, y_reg, reg_mask, groups, is_anchor, sample_qual, X_test_seq, X_test_tab, test_ids, test_sessions

# =====================================================================
# 3. THREE ELECTRODE-PRESERVING NEURAL NETS (ALL WITH 11-TARGET REG_HEAD)
# =====================================================================
class AuraDataset(Dataset):
    def __init__(self, X_seq, X_tab, y_cls=None, y_reg=None, reg_mask=None, sample_w=None, is_train=True):
        self.X_seq, self.X_tab = X_seq, X_tab
        self.y_cls, self.y_reg, self.reg_mask, self.sample_w = y_cls, y_reg, reg_mask, sample_w
        self.is_train = is_train

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        x, tab = self.X_seq[idx].copy(), self.X_tab[idx].copy()
        if self.is_train:
            if random.random() < 0.55:
                x = np.roll(x, shift=random.randint(-8, 8), axis=-1)
            if random.random() < 0.45:
                x = x * np.random.uniform(0.90, 1.10, size=(x.shape[0], 1)).astype(np.float32) + np.random.normal(0.0, 0.02, size=x.shape).astype(np.float32)
            if random.random() < 0.25:
                t0 = random.randint(0, x.shape[1] - 16)
                x[:, t0 : t0 + 16] = 0.0
        if self.y_cls is None:
            return torch.tensor(x, dtype=torch.float32), torch.tensor(tab, dtype=torch.float32)
        return (
            torch.tensor(x, dtype=torch.float32), torch.tensor(tab, dtype=torch.float32),
            torch.tensor(self.y_cls[idx], dtype=torch.long),
            torch.tensor(self.y_reg[idx], dtype=torch.float32),
            torch.tensor(self.reg_mask[idx], dtype=torch.float32),
            torch.tensor(self.sample_w[idx], dtype=torch.float32)
        )

class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(channels, channels // reduction), nn.GELU(), nn.Linear(channels // reduction, channels), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean(dim=-1)).unsqueeze(-1)

class AuraHybridNetV1Upgraded(nn.Module):
    """Model 1: Exact V16/V32 Winner (1D-CNN + SE + Transformer + 11-target reg_head)."""
    def __init__(self, in_channels=66, tab_dim=828, d_model=128, num_classes=6, num_reg=NUM_REG):
        super().__init__()
        self.inst_norm_v1 = nn.InstanceNorm1d(36, affine=True)
        self.bn_v3 = nn.BatchNorm1d(in_channels - 36)
        self.conv_k3 = nn.Conv1d(in_channels, d_model // 2, kernel_size=3, padding=1)
        self.conv_k7 = nn.Conv1d(in_channels, d_model // 4, kernel_size=7, padding=3)
        self.conv_k15 = nn.Conv1d(in_channels, d_model // 4, kernel_size=15, padding=7)
        self.cnn_encoder = nn.Sequential(
            nn.BatchNorm1d(d_model), nn.GELU(), SEBlock1D(d_model), nn.Dropout(0.15), nn.MaxPool1d(2, 2),
            nn.Conv1d(d_model, d_model, kernel_size=5, padding=2), nn.BatchNorm1d(d_model), nn.GELU(), SEBlock1D(d_model), nn.MaxPool1d(2, 2)
        )
        self.pos_emb = nn.Parameter(torch.randn(1, 48, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4, dim_feedforward=d_model * 2, dropout=0.20, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.attn_pool = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1))
        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim), nn.Dropout(0.25), nn.Linear(tab_dim, 160), nn.BatchNorm1d(160), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(160, 96), nn.BatchNorm1d(96), nn.GELU()
        )
        fused_dim = d_model * 2 + 96
        self.reg_head = nn.Linear(fused_dim, num_reg)
        self.cls_head = nn.Sequential(nn.LayerNorm(fused_dim + num_reg), nn.Dropout(0.30), nn.Linear(fused_dim + num_reg, 128), nn.GELU(), nn.Linear(128, num_classes))

    def forward(self, x_seq, x_tab):
        x = torch.cat([self.inst_norm_v1(x_seq[:, :36, :]), self.bn_v3(x_seq[:, 36:, :])], dim=1)
        x = self.cnn_encoder(torch.cat([self.conv_k3(x), self.conv_k7(x), self.conv_k15(x)], dim=1))
        seq = self.transformer(x.permute(0, 2, 1) + self.pos_emb)
        w = F.softmax(self.attn_pool(seq), dim=1)
        fused = torch.cat([torch.sum(seq * w, dim=1), torch.max(seq, dim=1)[0], self.tab_mlp(x_tab)], dim=-1)
        reg_p = self.reg_head(fused)
        return self.cls_head(torch.cat([fused, reg_p], dim=-1)), reg_p

class InceptionWaveBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        mid = out_c // 4
        self.b1, self.b2 = nn.Conv1d(in_c, mid, 1), nn.Conv1d(in_c, mid, 5, padding=2)
        self.b3, self.b4 = nn.Conv1d(in_c, mid, 11, padding=5), nn.Conv1d(in_c, mid, 23, padding=11)
        self.bn = nn.BatchNorm1d(out_c)
        self.short = nn.Conv1d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
    def forward(self, x):
        return F.gelu(self.bn(torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)) + self.short(x))

class InceptionBiGRUNetV3(nn.Module):
    """Model 2: Exact V16/V32 Winner (Un-instanced 66-ch Inception + BiGRU + 11-target reg_head)."""
    def __init__(self, in_channels=66, tab_dim=828, num_classes=6, num_reg=NUM_REG):
        super().__init__()
        self.inc1, self.pool1 = InceptionWaveBlock(in_channels, 96), nn.AvgPool1d(2, 2)
        self.inc2, self.pool2 = InceptionWaveBlock(96, 128), nn.AvgPool1d(2, 2)
        self.inc3 = InceptionWaveBlock(128, 128)
        self.bigru = nn.GRU(128, 64, num_layers=1, batch_first=True, bidirectional=True)
        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim), nn.Dropout(0.25), nn.Linear(tab_dim, 160), nn.BatchNorm1d(160), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(160, 96), nn.GELU()
        )
        fused_dim = 128 * 3 + 96
        self.reg_head = nn.Linear(fused_dim, num_reg)
        self.cls_head = nn.Sequential(nn.LayerNorm(fused_dim + num_reg), nn.Dropout(0.30), nn.Linear(fused_dim + num_reg, 128), nn.GELU(), nn.Linear(128, num_classes))

    def forward(self, x_seq, x_tab):
        h = self.inc3(self.pool2(self.inc2(self.pool1(self.inc1(x_seq)))))
        seq_g, _ = self.bigru(h.permute(0, 2, 1))
        h = h + seq_g.permute(0, 2, 1)
        fused = torch.cat([h.mean(dim=-1), h.std(dim=-1), h.max(dim=-1)[0], self.tab_mlp(x_tab)], dim=-1)
        reg_p = self.reg_head(fused)
        return self.cls_head(torch.cat([fused, reg_p], dim=-1)), reg_p

class DilatedResBlock1D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        mid = out_c // 3
        self.d1 = nn.Conv1d(in_c, mid, kernel_size=3, stride=stride, padding=1, dilation=1)
        self.d2 = nn.Conv1d(in_c, mid, kernel_size=3, stride=stride, padding=2, dilation=2)
        self.d4 = nn.Conv1d(in_c, out_c - 2 * mid, kernel_size=3, stride=stride, padding=4, dilation=4)
        self.bn = nn.BatchNorm1d(out_c)
        self.se = SEBlock1D(out_c)
        self.short = nn.Conv1d(in_c, out_c, 1, stride=stride) if (in_c != out_c or stride != 1) else nn.Identity()

    def forward(self, x):
        h = F.gelu(self.bn(torch.cat([self.d1(x), self.d2(x), self.d4(x)], dim=1)))
        return F.gelu(self.se(h) + self.short(x))

class MultiScaleResBiGRUNet(nn.Module):
    """Model 3: Electrode-Preserving Multi-Dilation ResNet + 2-Layer BiGRU + Attention + 11-target reg_head."""
    def __init__(self, in_channels=66, tab_dim=828, num_classes=6, num_reg=NUM_REG):
        super().__init__()
        self.res1 = DilatedResBlock1D(in_channels, 96, stride=2)
        self.res2 = DilatedResBlock1D(96, 128, stride=2)
        self.res3 = DilatedResBlock1D(128, 128, stride=1)
        self.bigru = nn.GRU(128, 64, num_layers=2, batch_first=True, bidirectional=True, dropout=0.20)
        self.attn_pool = nn.Sequential(nn.Linear(128, 64), nn.Tanh(), nn.Linear(64, 1))
        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim), nn.Dropout(0.25), nn.Linear(tab_dim, 160), nn.BatchNorm1d(160), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(160, 96), nn.BatchNorm1d(96), nn.GELU()
        )
        fused_dim = 128 * 2 + 96
        self.reg_head = nn.Linear(fused_dim, num_reg)
        self.cls_head = nn.Sequential(nn.LayerNorm(fused_dim + num_reg), nn.Dropout(0.30), nn.Linear(fused_dim + num_reg, 128), nn.GELU(), nn.Linear(128, num_classes))

    def forward(self, x_seq, x_tab):
        h = self.res3(self.res2(self.res1(x_seq)))
        seq, _ = self.bigru(h.permute(0, 2, 1))
        seq = seq + h.permute(0, 2, 1)
        w = F.softmax(self.attn_pool(seq), dim=1)
        fused = torch.cat([torch.sum(seq * w, dim=1), torch.max(seq, dim=1)[0], self.tab_mlp(x_tab)], dim=-1)
        reg_p = self.reg_head(fused)
        return self.cls_head(torch.cat([fused, reg_p], dim=-1)), reg_p

class QualityWeightedAuraLoss(nn.Module):
    """Exact V16/V32 AuraCompositeLoss with soft optical-agreement sample weights."""
    def __init__(self, num_classes=6, smoothing=0.06, gamma=2.0, f1_w=0.35, reg_w=0.25):
        super().__init__()
        self.num_classes, self.smoothing, self.gamma, self.f1_w, self.reg_w = num_classes, smoothing, gamma, f1_w, reg_w

    def forward(self, logits, y_cls, reg_p, y_reg, reg_m, sw):
        probs, log_probs = F.softmax(logits, dim=-1), F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            true_dist = torch.full_like(log_probs, self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, y_cls.unsqueeze(1), 1.0 - self.smoothing)
        focal_i = torch.sum(-true_dist * ((1.0 - probs) ** self.gamma) * log_probs, dim=-1)
        focal = torch.sum(focal_i * sw) / (torch.sum(sw) + 1e-6)

        one_hot = F.one_hot(y_cls, num_classes=self.num_classes).float()
        sw_col = sw.unsqueeze(1)
        tp = torch.sum(probs * one_hot * sw_col, dim=0)
        fp = torch.sum(probs * (1.0 - one_hot) * sw_col, dim=0)
        fn = torch.sum((1.0 - probs) * one_hot * sw_col, dim=0)
        soft_f1 = (2.0 * tp + 1e-5) / (2.0 * tp + fp + fn + 1e-5)
        loss = (1.0 - self.f1_w) * focal + self.f1_w * (1.0 - torch.mean(soft_f1))

        valid = torch.sum(reg_m)
        if valid > 0:
            loss = loss + self.reg_w * (torch.sum(F.smooth_l1_loss(reg_p, y_reg, reduction='none') * reg_m) / (valid + 1e-5))
        return loss

def predict_with_tta(model, loader, device):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for batch in loader:
            bx_s, bx_t = batch[0].to(device), batch[1].to(device)
            l0, _ = model(bx_s, bx_t)
            l1, _ = model(torch.roll(bx_s, shifts=-5, dims=-1), bx_t)
            l2, _ = model(torch.roll(bx_s, shifts=5, dims=-1), bx_t)
            p = 0.50 * F.softmax(l0, dim=-1) + 0.25 * F.softmax(l1, dim=-1) + 0.25 * F.softmax(l2, dim=-1)
            probs_list.append(p.cpu().numpy())
    return np.concatenate(probs_list, axis=0)

def train_v32_snapshot(model, trn_loader, val_loader, tst_loader, criterion):
    """Exact V32 Final-3-Epoch Cosine Probability Snapshots (Zero BN Corruption)."""
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1.5e-2)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    val_sum, tst_sum = None, None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for bx_s, bx_t, by_c, by_r, bm, bsw in trn_loader:
            bx_s, bx_t = bx_s.to(DEVICE), bx_t.to(DEVICE)
            by_c, by_r, bm, bsw = by_c.to(DEVICE), by_r.to(DEVICE), bm.to(DEVICE), bsw.to(DEVICE)
            opt.zero_grad()
            logits, reg_p = model(bx_s, bx_t)
            loss = criterion(logits, by_c, reg_p, by_r, bm, bsw)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()

        if epoch in SNAP_WEIGHTS:
            w = SNAP_WEIGHTS[epoch]
            vp = predict_with_tta(model, val_loader, DEVICE)
            tp = predict_with_tta(model, tst_loader, DEVICE)
            val_sum = w * vp if val_sum is None else val_sum + w * vp
            tst_sum = w * tp if tst_sum is None else tst_sum + w * tp

    return val_sum, tst_sum

def calibrate_session_logits(probs, session_ids, power=0.45, n_iter=4):
    cal_probs = probs.copy()
    for s_id in np.unique(session_ids):
        mask = (session_ids == s_id)
        if np.sum(mask) < 12:
            continue
        P = cal_probs[mask].copy() + 1e-6
        for _ in range(n_iter):
            P = P / np.sum(P, axis=1, keepdims=True)
            P = P * ((1.0 / 6.0) / (np.mean(P, axis=0, keepdims=True) + 1e-6)) ** power
            P = P / np.sum(P, axis=1, keepdims=True)
        cal_probs[mask] = P
    return cal_probs

# =====================================================================
# 4. 2-REPEAT x 5-FOLD CROSS-SUBJECT BAGGING (60 MODELS ON V32 CORE)
# =====================================================================
(X_seq, X_tab, y_cls, y_reg, reg_mask, groups, is_anchor, sample_qual,
 X_test_seq, X_test_tab, test_ids, test_sessions) = load_v33_datasets(DATA_DIR)

N_REPEATS = len(SPLIT_CONFIGS)
TOTAL_SPLITS = N_REPEATS * N_FOLDS

oof_v1 = np.zeros((len(X_seq), 6), np.float32)
oof_v3 = np.zeros((len(X_seq), 6), np.float32)
oof_v4 = np.zeros((len(X_seq), 6), np.float32)
oof_lgb = np.zeros((len(X_seq), 6), np.float32)
oof_cat = np.zeros((len(X_seq), 6), np.float32)
oof_xgb = np.zeros((len(X_seq), 6), np.float32)

test_v1 = np.zeros((len(X_test_seq), 6), np.float32)
test_v3 = np.zeros((len(X_test_seq), 6), np.float32)
test_v4 = np.zeros((len(X_test_seq), 6), np.float32)
test_lgb = np.zeros((len(X_test_seq), 6), np.float32)
test_cat = np.zeros((len(X_test_seq), 6), np.float32)
test_xgb = np.zeros((len(X_test_seq), 6), np.float32)

criterion = QualityWeightedAuraLoss()
clean_mask = (is_anchor == 1)
y_clean = y_cls[clean_mask]

for rep_idx, cfg in enumerate(SPLIT_CONFIGS):
    print(f"\n================ REPEAT {rep_idx + 1} / {N_REPEATS} (shuffle={cfg['shuffle']}) ================")
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=cfg['shuffle'], random_state=cfg['random_state'])

    rep_v1 = np.zeros((len(X_seq), 6), np.float32)
    rep_v3 = np.zeros((len(X_seq), 6), np.float32)
    rep_v4 = np.zeros((len(X_seq), 6), np.float32)
    rep_lgb = np.zeros((len(X_seq), 6), np.float32)
    rep_cat = np.zeros((len(X_seq), 6), np.float32)
    rep_xgb = np.zeros((len(X_seq), 6), np.float32)

    for fold, (trn_idx, val_idx) in enumerate(sgkf.split(X_seq, y_cls, groups)):
        seed_now = cfg['seed_base'] + fold * 17
        seed_everything(seed_now)

        clean_val_idx = val_idx[is_anchor[val_idx] == 1]
        y_val_clean = y_cls[clean_val_idx]
        trn_sw = sample_qual[trn_idx]

        scaler = StandardScaler()
        X_tab_trn = np.clip(scaler.fit_transform(X_tab[trn_idx]), -8.0, 8.0)
        X_tab_val = np.clip(scaler.transform(X_tab[val_idx]), -8.0, 8.0)
        X_tab_val_clean = np.clip(scaler.transform(X_tab[clean_val_idx]), -8.0, 8.0)
        X_tab_tst = np.clip(scaler.transform(X_test_tab), -8.0, 8.0)

        scout = LGBMClassifier(n_estimators=220, learning_rate=0.04, max_depth=4, num_leaves=15, subsample=0.8, colsample_bytree=0.6, random_state=seed_now, n_jobs=-1, verbose=-1)
        scout.fit(X_tab_trn, y_cls[trn_idx], sample_weight=trn_sw)
        keep_cols = np.argsort(scout.feature_importances_)[::-1][:int(X_tab.shape[1] * 0.75)]

        # 1. LightGBM
        lgb_m = LGBMClassifier(
            n_estimators=1600, learning_rate=0.022, max_depth=5, num_leaves=22, min_child_samples=28,
            min_split_gain=0.01, subsample=0.80, subsample_freq=1, colsample_bytree=0.60,
            reg_alpha=2.0, reg_lambda=4.0, random_state=100 + seed_now, n_jobs=-1, verbose=-1
        )
        lgb_m.fit(X_tab_trn[:, keep_cols], y_cls[trn_idx], sample_weight=trn_sw, eval_set=[(X_tab_val_clean[:, keep_cols], y_val_clean)], callbacks=[early_stopping(100, verbose=False)])
        lgb_vp = lgb_m.predict_proba(X_tab_val[:, keep_cols])
        lgb_tp = lgb_m.predict_proba(X_tab_tst[:, keep_cols])
        rep_lgb[val_idx] = lgb_vp; test_lgb += lgb_tp / TOTAL_SPLITS

        # 2. CatBoost
        if HAS_CATBOOST:
            cat_m = CatBoostClassifier(iterations=1600, learning_rate=0.032, depth=5, l2_leaf_reg=6.0, random_seed=100 + seed_now, task_type='CPU', verbose=False)
            cat_m.fit(X_tab_trn[:, keep_cols], y_cls[trn_idx], sample_weight=trn_sw, eval_set=(X_tab_val_clean[:, keep_cols], y_val_clean), early_stopping_rounds=100, verbose=False)
            cat_vp = cat_m.predict_proba(X_tab_val[:, keep_cols])
            cat_tp = cat_m.predict_proba(X_tab_tst[:, keep_cols])
            rep_cat[val_idx] = cat_vp; test_cat += cat_tp / TOTAL_SPLITS
        else:
            rep_cat[val_idx] = lgb_vp.copy(); test_cat += lgb_tp / TOTAL_SPLITS

        # 3. XGBoost
        if HAS_XGB:
            xgb_m = XGBClassifier(
                n_estimators=1400, learning_rate=0.025, max_depth=5, subsample=0.80, colsample_bytree=0.58,
                reg_alpha=2.0, reg_lambda=4.5, min_child_weight=5, random_state=100 + seed_now, n_jobs=-1,
                eval_metric='mlogloss', early_stopping_rounds=90
            )
            xgb_m.fit(X_tab_trn[:, keep_cols], y_cls[trn_idx], sample_weight=trn_sw, eval_set=[(X_tab_val_clean[:, keep_cols], y_val_clean)], verbose=False)
            xgb_vp = xgb_m.predict_proba(X_tab_val[:, keep_cols])
            xgb_tp = xgb_m.predict_proba(X_tab_tst[:, keep_cols])
            rep_xgb[val_idx] = xgb_vp; test_xgb += xgb_tp / TOTAL_SPLITS
        else:
            rep_xgb[val_idx] = 0.5 * (rep_lgb[val_idx] + rep_cat[val_idx]); test_xgb += lgb_tp / TOTAL_SPLITS

        # 4, 5 & 6. Neural Trio (All with 11-Target reg_head & Final-3-Epoch Snapshots)
        trn_loader = DataLoader(
            AuraDataset(X_seq[trn_idx], X_tab_trn, y_cls[trn_idx], y_reg[trn_idx], reg_mask[trn_idx], trn_sw, is_train=True),
            batch_size=BATCH_SIZE, shuffle=True, drop_last=True
        )
        val_loader = DataLoader(AuraDataset(X_seq[val_idx], X_tab_val, is_train=False), batch_size=BATCH_SIZE, shuffle=False)
        tst_loader = DataLoader(AuraDataset(X_test_seq, X_tab_tst, is_train=False), batch_size=BATCH_SIZE, shuffle=False)

        m1 = AuraHybridNetV1Upgraded(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
        vp1, tp1 = train_v32_snapshot(m1, trn_loader, val_loader, tst_loader, criterion)
        rep_v1[val_idx] = vp1; test_v1 += tp1 / TOTAL_SPLITS

        m2 = InceptionBiGRUNetV3(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
        vp2, tp2 = train_v32_snapshot(m2, trn_loader, val_loader, tst_loader, criterion)
        rep_v3[val_idx] = vp2; test_v3 += tp2 / TOTAL_SPLITS

        m3 = MultiScaleResBiGRUNet(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
        vp3, tp3 = train_v32_snapshot(m3, trn_loader, val_loader, tst_loader, criterion)
        rep_v4[val_idx] = vp3; test_v4 += tp3 / TOTAL_SPLITS

        print(
            f"  [R{rep_idx+1} F{fold+1}] Val F1 -> LGB: {f1_score(y_val_clean, np.argmax(rep_lgb[clean_val_idx], 1), average='macro'):.4f} | "
            f"CAT: {f1_score(y_val_clean, np.argmax(rep_cat[clean_val_idx], 1), average='macro'):.4f} | "
            f"V1: {f1_score(y_val_clean, np.argmax(rep_v1[clean_val_idx], 1), average='macro'):.4f} | "
            f"V3: {f1_score(y_val_clean, np.argmax(rep_v3[clean_val_idx], 1), average='macro'):.4f} | "
            f"ResBiGRU: {f1_score(y_val_clean, np.argmax(rep_v4[clean_val_idx], 1), average='macro'):.4f}"
        )

    oof_v1 += rep_v1 / N_REPEATS
    oof_v3 += rep_v3 / N_REPEATS
    oof_v4 += rep_v4 / N_REPEATS
    oof_lgb += rep_lgb / N_REPEATS
    oof_cat += rep_cat / N_REPEATS
    oof_xgb += rep_xgb / N_REPEATS

# =====================================================================
# 5. NEURAL-PROTECTED 6-MODEL BLENDING & EXACT V32 CALIBRATION
# =====================================================================
print("\n--- 2x5-Fold Bagged Individual Clean Anchor OOF Macro-F1 ---")
print(f"  10-Model LightGBM:         {f1_score(y_clean, np.argmax(oof_lgb[clean_mask], 1), average='macro'):.4f}")
print(f"  10-Model CatBoost:         {f1_score(y_clean, np.argmax(oof_cat[clean_mask], 1), average='macro'):.4f}")
print(f"  10-Model XGBoost:          {f1_score(y_clean, np.argmax(oof_xgb[clean_mask], 1), average='macro'):.4f}")
print(f"  10-Model V1-Hybrid:        {f1_score(y_clean, np.argmax(oof_v1[clean_mask], 1), average='macro'):.4f}")
print(f"  10-Model V3-Inception:     {f1_score(y_clean, np.argmax(oof_v3[clean_mask], 1), average='macro'):.4f}")
print(f"  10-Model MultiScaleResGRU: {f1_score(y_clean, np.argmax(oof_v4[clean_mask], 1), average='macro'):.4f}")

# Step A: Internal Neural Trio Weights (V4 allowed to be 0.0 if it doesn't add OOF lift)
best_nn_f1, best_nn_w = 0.0, (0.40, 0.35, 0.25)
for a1 in np.linspace(0.20, 0.60, 9):
    for a2 in np.linspace(0.20, 0.60, 9):
        a3 = round(1.0 - a1 - a2, 4)
        if a3 < -1e-6 or a3 > 0.45:
            continue
        a3 = max(0.0, a3)
        s = f1_score(y_clean, np.argmax(a1 * oof_v1[clean_mask] + a2 * oof_v3[clean_mask] + a3 * oof_v4[clean_mask], axis=1), average='macro')
        if s > best_nn_f1:
            best_nn_f1, best_nn_w = s, (a1, a2, a3)

oof_nn = best_nn_w[0] * oof_v1 + best_nn_w[1] * oof_v3 + best_nn_w[2] * oof_v4
test_nn = best_nn_w[0] * test_v1 + best_nn_w[1] * test_v3 + best_nn_w[2] * test_v4

# Step B: Internal Tree Trio Weights (XGB allowed to be 0.0 if it doesn't add OOF lift)
best_tr_f1, best_tr_w = 0.0, (0.45, 0.40, 0.15)
for t1 in np.linspace(0.25, 0.65, 9):
    for t2 in np.linspace(0.25, 0.65, 9):
        t3 = round(1.0 - t1 - t2, 4)
        if t3 < -1e-6 or t3 > 0.35:
            continue
        t3 = max(0.0, t3)
        s = f1_score(y_clean, np.argmax(t1 * oof_lgb[clean_mask] + t2 * oof_cat[clean_mask] + t3 * oof_xgb[clean_mask], axis=1), average='macro')
        if s > best_tr_f1:
            best_tr_f1, best_tr_w = s, (t1, t2, t3)

oof_trees = best_tr_w[0] * oof_lgb + best_tr_w[1] * oof_cat + best_tr_w[2] * oof_xgb
test_trees = best_tr_w[0] * test_lgb + best_tr_w[1] * test_cat + best_tr_w[2] * test_xgb

# Step C: Protected Neural-Heavy Blend (55%-80% Neural Share matching V32 0.67681)
best_raw_f1, best_nn_share = 0.0, 0.65
for w_nn in np.linspace(0.55, 0.80, 11):
    s = f1_score(y_clean, np.argmax(w_nn * oof_nn[clean_mask] + (1.0 - w_nn) * oof_trees[clean_mask], axis=1), average='macro')
    if s > best_raw_f1:
        best_raw_f1, best_nn_share = s, w_nn

oof_blend = best_nn_share * oof_nn + (1.0 - best_nn_share) * oof_trees
test_blend = best_nn_share * test_nn + (1.0 - best_nn_share) * test_trees

best_cal_f1, best_power, oof_cal = best_raw_f1, 0.0, oof_blend.copy()
for pwr in [0.20, 0.30, 0.40, 0.50]:
    cand = calibrate_session_logits(oof_blend, groups, power=pwr, n_iter=4)
    s = f1_score(y_clean, np.argmax(cand[clean_mask], axis=1), average='macro')
    if s > best_cal_f1:
        best_cal_f1, best_power, oof_cal = s, pwr, cand

test_cal = calibrate_session_logits(test_blend, test_sessions, power=best_power, n_iter=4) if best_power > 0.0 else test_blend

class_mult, tuned_f1 = np.ones(6, dtype=np.float32), best_cal_f1
for _ in range(3):
    for c in range(6):
        for m in np.linspace(0.90, 1.12, 23):
            cand = class_mult.copy(); cand[c] = m
            s = f1_score(y_clean, np.argmax(oof_cal[clean_mask] * cand[None, :], axis=1), average='macro')
            if s > tuned_f1:
                tuned_f1, class_mult = s, cand

oof_final_preds = np.argmax(oof_cal[clean_mask] * class_mult[None, :], axis=1)
print("\n========================================================")
print(f"Clean OOF Macro-F1 (3-NN Trio Alone):       {best_nn_f1:.4f} | NN Weights (V1, V3, ResGRU): {np.round(best_nn_w, 2)}")
print(f"Clean OOF Macro-F1 (3-Tree Trio Alone):     {best_tr_f1:.4f} | Tree Weights (LGB, CAT, XGB): {np.round(best_tr_w, 2)}")
print(f"Clean OOF Macro-F1 (V33 60-Model Blend):    {best_raw_f1:.4f} | Neural Share: {best_nn_share:.2f}")
print(f"Clean OOF Macro-F1 (+Session Calibration):  {best_cal_f1:.4f} | Power: {best_power}")
print(f"Clean OOF Macro-F1 (+Tuned Final):          {tuned_f1:.4f} | Mult: {np.round(class_mult, 3)}")
print("========================================================")
print(classification_report(y_clean, oof_final_preds, target_names=GESTURES))
print("Confusion Matrix (Rows=True, Cols=Pred):")
print(pd.DataFrame(confusion_matrix(y_clean, oof_final_preds), index=GESTURES, columns=GESTURES))

final_labels = [INV_GESTURE_MAP[i] for i in np.argmax(test_cal * class_mult[None, :], axis=1)]
sample_sub_files = glob.glob(os.path.join(DATA_DIR, "**", "sample_submission*.csv"), recursive=True)
if sample_sub_files:
    sub = pd.read_csv(sample_sub_files[0])
    id_col, target_col = sub.columns[0], sub.columns[1]
    if len(sub) == len(final_labels) and len(set(test_ids)) == len(test_ids):
        sub[target_col] = sub[id_col].map(dict(zip(test_ids, final_labels))).fillna(pd.Series(final_labels))
    else:
        sub[target_col] = final_labels
else:
    sub = pd.DataFrame({'action_id': test_ids, 'gesture': final_labels})

sub.to_csv("submission.csv", index=False)
sub.to_csv("/kaggle/working/submission.csv", index=False)

with open("submission.csv", "rb") as f:
    sub_hash = hashlib.sha256(f.read()).hexdigest()[:12]

print(f"\n=== DONE: V33-HUMNOTCHPRO READY (SHA256: {sub_hash}) ===")
print(sub.iloc[:, 1].value_counts())
# === END OF COMPLETE V33 SCRIPT ===